# 🩸 Blood Group Prediction from Fingerprint Images
### Deep Learning Project — VGG16 Transfer Learning (Improved Version)

**Project Flow:**
1. Import Libraries
2. Load & Explore Dataset
3. Data Augmentation & Preprocessing
4. Model Architecture (VGG16 + Fine-Tuning)
5. Training with Callbacks
6. Fine-Tuning Phase
7. Training Curves
8. Evaluation & Metrics
9. Save Model
10. Single Image Prediction

---
## 📦 Step 1: Import Libraries

In [ ]:
# ─── Standard Library ────────────────────────────────────────────────────────
import os
import glob
import json
import warnings
warnings.filterwarnings('ignore')      # suppress minor TF/Keras warnings

# ─── Data Handling ───────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ─── Visualisation ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ─── Scikit-learn ────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# ─── TensorFlow / Keras ──────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image as keras_image
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense, Dropout, BatchNormalization, GlobalAveragePooling2D
)
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.optimizers import Adam

# ─── Reproducibility Seed ────────────────────────────────────────────────────
# Setting seeds ensures our random operations (shuffling, weight init)
# give the same result every run → experiments are reproducible.
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow version : {tf.__version__}")
print(f"GPU available      : {tf.config.list_physical_devices('GPU')}")

---
## 📂 Step 2: Load & Explore Dataset

In [ ]:
# ─── Configuration ───────────────────────────────────────────────────────────
# UPDATE this path to wherever your dataset folder lives.
#
# Expected folder structure:
#   datasets/
#       A+/   cluster_0_1.BMP   cluster_0_2.BMP  ...
#       A-/   ...
#       AB+/  ...
#       AB-/  ...
#       B+/   ...
#       B-/   ...
#       O+/   ...
#       O-/   ...
#
DATASET_PATH = r'D:\dataset_blood_group'    # <-- UPDATE to your actual dataset path
IMG_SIZE     = (224, 224)     # VGG16 canonical input size (224x224)
BATCH_SIZE   = 32
EPOCHS       = 30             # Max epochs; EarlyStopping will cut short if needed
NUM_CLASSES  = 8              # A+  A-  AB+  AB-  B+  B-  O+  O-

# ─── Gather All Image Paths & Derive Labels from Folder Names ────────────────
# glob.glob with recursive=True walks through all sub-directories.
filepaths = glob.glob(os.path.join(DATASET_PATH, '**', '*.*'), recursive=True)

# Label = name of the immediate parent folder (e.g. 'A+', 'O-')
labels = [os.path.basename(os.path.dirname(fp)) for fp in filepaths]

# ─── Build a Tidy DataFrame ──────────────────────────────────────────────────
df = pd.DataFrame({'Filepath': filepaths, 'Label': labels})
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)   # shuffle rows

print(f"Total images found : {len(df)}")
print(f"Classes detected   : {sorted(df.Label.unique())}")
print("\nSample rows:")
display(df.head())

In [ ]:
# ─── Class Distribution Plot ─────────────────────────────────────────────────
# Always check for class imbalance BEFORE training.
# If one class has 3x more images than another, the model may ignore the
# minority class and still get high accuracy — that would be misleading.

counts = df.Label.value_counts().sort_index()

plt.figure(figsize=(9, 4))
sns.barplot(x=counts.index, y=counts.values, palette='Blues_d')
plt.title('Class Distribution — Blood Groups', fontsize=14)
plt.xlabel('Blood Group')
plt.ylabel('Number of Images')
for i, v in enumerate(counts.values):
    plt.text(i, v + 1, str(v), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

print(counts)

In [ ]:
# ─── Visualise One Sample Image per Class ────────────────────────────────────

# Get only the valid 8 blood group classes
VALID_CLASSES = ['A+', 'A-', 'AB+', 'AB-', 'B+', 'B-', 'O+', 'O-']

# Filter df to keep only valid labels (removes any junk folders)
df = df[df.Label.isin(VALID_CLASSES)].reset_index(drop=True)

# Confirm what classes are present
unique_classes = sorted(df.Label.unique())
print(f"Classes found: {unique_classes}  → count: {len(unique_classes)}")

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()

for idx, blood_group in enumerate(unique_classes):
    sample_path = df[df.Label == blood_group].iloc[0]['Filepath']
    img = plt.imread(sample_path)
    axes[idx].imshow(img, cmap='gray')
    axes[idx].set_title(f'Blood Group: {blood_group}', fontsize=11)
    axes[idx].axis('off')

plt.suptitle('Sample Fingerprint per Blood Group', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## ✂️ Step 3: Train / Validation / Test Split & Data Augmentation

In [ ]:
# ─── Stratified 3-Way Split (70 / 15 / 15) ───────────────────────────────────
#
# Why stratify?
#   stratify=df.Label ensures each split contains the SAME proportion of each
#   blood group. Without this, by chance you could get 0 samples of a rare class
#   in validation → misleadingly high accuracy on that class.
#
# Why 3 splits?
#   Train   → model sees this data and learns
#   Val     → used DURING training to tune hyperparameters & stop early
#   Test    → ONLY looked at once, at the very end → unbiased final score

train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=SEED, stratify=df.Label
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df.Label
)

print(f"Train  split : {len(train_df):>5} images  ({len(train_df)/len(df)*100:.0f}%)")
print(f"Val    split : {len(val_df):>5} images  ({len(val_df)/len(df)*100:.0f}%)")
print(f"Test   split : {len(test_df):>5} images  ({len(test_df)/len(df)*100:.0f}%)")

In [ ]:
# ─── Data Generators with Augmentation ───────────────────────────────────────
#
# TRAIN  → preprocessing + augmentation
# VAL / TEST → preprocessing ONLY (no augmentation)
#
# Why augment training data?
#   Fingerprint datasets are often small (a few hundred images per class).
#   Augmentation creates synthetically varied versions of each image:
#   slightly rotated, shifted, zoomed, or flipped.
#   This forces the model to learn features that are robust to these variations
#   instead of memorising exact pixel positions → REDUCES OVERFITTING.
#
# Why NOT augment val/test?
#   We want a stable, reproducible metric. Augmenting test data would make
#   each evaluation give slightly different numbers.

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # VGG16-specific mean subtraction
    rotation_range=15,        # rotate up to ±15 degrees
    width_shift_range=0.10,   # horizontal shift up to 10% of image width
    height_shift_range=0.10,  # vertical shift up to 10% of image height
    zoom_range=0.10,          # zoom in/out by up to 10%
    horizontal_flip=True,     # randomly mirror images (fingerprints are ~symmetric)
    fill_mode='nearest'       # fill new pixels after shift/rotate with nearest value
)

val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input   # ONLY normalise; NO augmentation
)

# ─── Flow generators from DataFrame ─────────────────────────────────────────
train_gen = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='Filepath',
    y_col='Label',
    target_size=IMG_SIZE,
    class_mode='categorical',  # one-hot encoded labels (required for softmax output)
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_gen = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='Filepath',
    y_col='Label',
    target_size=IMG_SIZE,
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=False    # keep order → predictions align with true labels
)

test_gen = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='Filepath',
    y_col='Label',
    target_size=IMG_SIZE,
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=False
)

# ─── Save class → index mapping (needed later for prediction) ─────────────────
CLASS_INDICES = train_gen.class_indices           # {'A+': 0, 'A-': 1, ...}
IDX_TO_CLASS  = {v: k for k, v in CLASS_INDICES.items()}  # reverse: {0: 'A+', ...}
print('Class → Index mapping:', CLASS_INDICES)

---
## 🏗️ Step 4: Model Architecture — VGG16 Transfer Learning

In [ ]:
# ─── Transfer Learning: Why VGG16? ───────────────────────────────────────────
#
# VGG16 was trained on ImageNet (1.2 million images, 1000 classes).
# Its convolutional layers have already learned to detect:
#   edges → curves → textures → parts → objects
#
# Fingerprint images share low-level structure (ridges, curves, texture)
# with ImageNet images → those pre-trained features transfer well.
#
# We REMOVE VGG16's original top (1000-class classifier) and add our own
# 8-class head tailored to blood groups.
#
# ADDED IMPROVEMENTS over original code:
#   • GlobalAveragePooling2D instead of flatten → fewer parameters, less overfitting
#   • BatchNormalization → stabilises training, lets us use higher LR
#   • Dropout(0.4 / 0.3) → regularisation to prevent overfitting
#   • Larger first Dense layer (256) → more capacity to learn fingerprint features

def build_model(num_classes: int, img_size: tuple):
    """Construct a VGG16-based transfer learning model."""

    # ── Load VGG16 base without its top classification layers ─────────────────
    base_model = VGG16(
        input_shape=(*img_size, 3),
        include_top=False,     # drop original FC layers
        weights='imagenet',    # pre-trained weights from ImageNet
        pooling=None           # we add our own pooling below
    )
    base_model.trainable = False   # FREEZE all base layers in Phase 1

    # ── Custom Classification Head ────────────────────────────────────────────
    # GlobalAveragePooling2D: averages spatial dimensions → output shape (batch, 512)
    # Much better than Flatten here because it greatly reduces parameters
    x = GlobalAveragePooling2D()(base_model.output)

    # First dense block
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)    # normalise activations after each Dense layer
    x = Dropout(0.4)(x)            # drop 40% of neurons → fights overfitting

    # Second dense block
    x = Dense(128, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    # Output layer: 8 neurons (one per blood group), softmax → probabilities sum to 1
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=outputs)
    return model, base_model


model, base_model = build_model(NUM_CLASSES, IMG_SIZE)

# ─── Compile ─────────────────────────────────────────────────────────────────
# Adam is the standard adaptive optimizer for CNNs.
# lr=1e-3 is the classic Adam default for Phase 1 (frozen base).
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',   # standard for multi-class classification
    metrics=['accuracy']
)

model.summary()

---
## 🔁 Step 5: Phase 1 Training — Frozen Base (Head Only)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks_phase1 = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=7,
        restore_best_weights=True,
        verbose=0
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=0
    ),
    ModelCheckpoint(
        filepath='best_model_phase1.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=0
    )
]

print("=" * 55)
print(" Phase 1 — Training classification head (base frozen)")
print("=" * 55)

history_phase1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks_phase1,
    verbose=2
)

---
## 🔧 Step 6: Phase 2 — Fine-Tuning (Unfreeze Top VGG Blocks)

In [ ]:
# ─── Fine-Tuning Strategy ─────────────────────────────────────────────────────
#
# After Phase 1 the custom head is well-trained.
# Now we UNFREEZE the top portion of VGG16 (last ~2 blocks) and train
# everything jointly at a MUCH lower learning rate (100x smaller).
#
# Why NOT unfreeze the whole network?
#   Early VGG16 layers (block1, block2) detect universal low-level features
#   (edges, corners) that apply to all images. Retraining them on our small
#   dataset would cause 'catastrophic forgetting' — losing those useful features.
#
# Why a low learning rate (1e-5)?
#   The base weights are already excellent; large updates would destroy them.
#   Small updates gently nudge the high-level features toward fingerprint patterns.

base_model.trainable = True   # first, unlock everything

# Re-freeze all layers except the last 8 (≈ VGG16 block4_conv3, block5_*)
FREEZE_UNTIL = len(base_model.layers) - 8
for layer in base_model.layers[:FREEZE_UNTIL]:
    layer.trainable = False

trainable_count = sum(1 for l in model.layers if l.trainable)
print(f"Trainable layers after fine-tuning setup : {trainable_count} / {len(model.layers)}")

# Recompile with a MUCH lower learning rate
model.compile(
    optimizer=Adam(learning_rate=1e-5),   # 100x smaller than Phase 1
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase2 = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=10,                   # more patience during fine-tuning
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        filepath='best_model_phase2.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print("=" * 55)
print(" Phase 2 — Fine-Tuning top VGG16 convolutional blocks")
print("=" * 55)

history_phase2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks_phase2
)

---
## 📈 Step 7: Training Curves (Both Phases)

In [ ]:
# ─── Combine Phase 1 + Phase 2 Histories ─────────────────────────────────────
# This gives us one continuous accuracy/loss curve for the full training run.

def combine_histories(h1, h2):
    """Concatenate two Keras History dicts into one."""
    combined = {}
    for key in h1.history:
        combined[key] = h1.history[key] + h2.history[key]
    return combined

all_history  = combine_histories(history_phase1, history_phase2)
epochs_range = range(1, len(all_history['accuracy']) + 1)
phase1_len   = len(history_phase1.history['accuracy'])   # epoch where FT starts

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Accuracy ──────────────────────────────────────────────────────────────
ax1.plot(epochs_range, all_history['accuracy'],     label='Train Accuracy', linewidth=2)
ax1.plot(epochs_range, all_history['val_accuracy'], label='Val Accuracy',   linewidth=2)
ax1.axvline(phase1_len, color='red', linestyle='--', alpha=0.6, label='Fine-tuning start')
ax1.set_title('Accuracy — Phase 1 → Phase 2', fontsize=13)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# ── Loss ──────────────────────────────────────────────────────────────────
ax2.plot(epochs_range, all_history['loss'],     label='Train Loss', linewidth=2)
ax2.plot(epochs_range, all_history['val_loss'], label='Val Loss',   linewidth=2)
ax2.axvline(phase1_len, color='red', linestyle='--', alpha=0.6, label='Fine-tuning start')
ax2.set_title('Loss — Phase 1 → Phase 2', fontsize=13)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Full Training History', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

---
## 🧪 Step 8: Final Evaluation on Held-Out Test Set

In [ ]:
# ─── Test Set Evaluation ─────────────────────────────────────────────────────
# CRITICAL: We only look at test data ONCE — right here at the very end.
# If you use test results to make any training decisions (tuning, re-running),
# the test set becomes contaminated and the score is no longer honest.

test_loss, test_acc = model.evaluate(test_gen, verbose=1)
print(f"\n{'='*40}")
print(f"  Final Test Accuracy : {test_acc * 100:.2f}%")
print(f"  Final Test Loss     : {test_loss:.4f}")
print(f"{'='*40}")

In [ ]:
# ─── Classification Report ───────────────────────────────────────────────────
# Precision = of all images predicted as class X, how many were actually X?
# Recall    = of all actual class X images, how many did we correctly predict?
# F1-Score  = harmonic mean of Precision and Recall (balanced metric)
#
# These per-class metrics are MUCH more informative than overall accuracy,
# especially when classes are imbalanced.

# Generate predictions on the test set
y_pred_probs  = model.predict(test_gen, verbose=1)          # shape: (N, 8)
y_pred_idx    = np.argmax(y_pred_probs, axis=1)             # index of highest prob
y_pred_labels = [IDX_TO_CLASS[i] for i in y_pred_idx]      # map to class names

# True labels — must use shuffle=False in test_gen for correct alignment!
y_true_labels = list(test_df.Label)

print("\nClassification Report (Test Set):")
print("─" * 55)
print(classification_report(
    y_true_labels, y_pred_labels,
    target_names=sorted(CLASS_INDICES.keys())
))

In [ ]:
# ─── Confusion Matrix ────────────────────────────────────────────────────────
# Rows = True label, Columns = Predicted label.
# Diagonal = correctly classified.  Off-diagonal = mistakes.
# Useful for identifying WHICH pairs of blood groups are being confused.

class_names = sorted(CLASS_INDICES.keys())
cm = confusion_matrix(y_true_labels, y_pred_labels, labels=class_names)

fig, ax = plt.subplots(figsize=(9, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title('Confusion Matrix — Test Set', fontsize=14)
plt.tight_layout()
plt.show()

---
## 💾 Step 9: Save the Model & Class Indices

In [ ]:
# ─── Save Trained Model ──────────────────────────────────────────────────────
# .keras  → modern TF2 recommended format
# .h5     → legacy HDF5 format, needed for older Flask/Python deployments

model.save('blood_group_vgg16_finetuned.keras')
print("Model saved → blood_group_vgg16_finetuned.keras")

model.save('blood_group_vgg16_finetuned.h5')
print("Model saved → blood_group_vgg16_finetuned.h5")

# ─── Save Class Index Mapping ────────────────────────────────────────────────
# Flask app / prediction script needs to know which index = which blood group.
# Save this alongside the model so they are always in sync.
with open('class_indices.json', 'w') as f:
    json.dump(CLASS_INDICES, f, indent=2)
print("Class indices saved → class_indices.json")

---
## 🔍 Step 10: Single Image Prediction (Inference)

In [ ]:
# ─── Reusable Prediction Function ────────────────────────────────────────────
# Encapsulating prediction logic into a function makes it easy to:
#   • Call from Flask app
#   • Reuse in batch prediction scripts
#   • Unit-test independently

def predict_blood_group(img_path: str, model, idx_to_class: dict,
                        img_size=(224, 224)) -> dict:
    """
    Predict blood group from a single fingerprint image.

    Parameters
    ----------
    img_path     : str   — path to fingerprint image file (.BMP / .JPG / .PNG)
    model        : keras Model — trained model
    idx_to_class : dict  — {0: 'A+', 1: 'A-', ...}  (reverse of class_indices)
    img_size     : tuple — must MATCH the img_size used during training

    Returns
    -------
    dict with keys:
        'label'         : predicted blood group (str)
        'confidence'    : confidence % of top prediction (float)
        'probabilities' : confidence % for every class (dict)
    """
    # 1. Load image and resize to match training dimensions
    img = keras_image.load_img(img_path, target_size=img_size)

    # 2. Convert PIL Image → numpy array: shape (H, W, C)
    img_array = keras_image.img_to_array(img)

    # 3. Add batch dimension: (H, W, C) → (1, H, W, C)
    img_array = np.expand_dims(img_array, axis=0)

    # 4. Apply the SAME preprocessing used during training!
    #    VGG16 preprocess_input subtracts ImageNet channel means.
    #    Skipping this step would give garbage predictions.
    img_array = preprocess_input(img_array)

    # 5. Forward pass → output shape: (1, 8) probabilities
    probs = model.predict(img_array, verbose=0)[0]   # shape (8,)

    # 6. Decode prediction
    pred_idx   = int(np.argmax(probs))
    pred_label = idx_to_class[pred_idx]
    confidence = float(probs[pred_idx]) * 100

    return {
        'label'        : pred_label,
        'confidence'   : confidence,
        'probabilities': {idx_to_class[i]: round(float(probs[i]) * 100, 2)
                          for i in range(len(probs))}
    }


# ─── Test on One Image from the Test Set ─────────────────────────────────────
TEST_IMG_PATH = test_df.iloc[0]['Filepath']   # use first test image
TRUE_LABEL    = test_df.iloc[0]['Label']

result = predict_blood_group(TEST_IMG_PATH, model, IDX_TO_CLASS)

# ─── Display Fingerprint + Prediction ────────────────────────────────────────
img_display = keras_image.load_img(TEST_IMG_PATH, target_size=IMG_SIZE)

plt.figure(figsize=(5, 5))
plt.imshow(img_display, cmap='gray')
plt.axis('off')
color = 'green' if result['label'] == TRUE_LABEL else 'red'
plt.title(
    f"Predicted : {result['label']}  ({result['confidence']:.1f}%)\n"
    f"True Label: {TRUE_LABEL}",
    fontsize=13, color=color
)
plt.tight_layout()
plt.show()

# ─── Confidence Bar Chart ─────────────────────────────────────────────────────
# Shows model certainty across all 8 blood groups.
# A well-trained model should have a tall bar for the correct class
# and near-zero bars for all others.

probs_sorted = dict(sorted(result['probabilities'].items()))
bar_colors   = ['orange' if k == result['label'] else 'steelblue'
                for k in probs_sorted.keys()]

plt.figure(figsize=(9, 3))
bars = plt.bar(probs_sorted.keys(), probs_sorted.values(), color=bar_colors)
plt.ylabel('Confidence (%)')
plt.title('Prediction Confidence per Blood Group (orange = predicted)')
plt.ylim(0, 110)
for bar in bars:
    h = bar.get_height()
    if h > 0.5:
        plt.text(bar.get_x() + bar.get_width()/2, h + 1,
                 f'{h:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## ✅ Project Summary

| Step | What We Did | Why It Matters |
|------|-------------|----------------|
| **Data split** | Stratified 70 / 15 / 15 | Equal class proportions in every split |
| **Augmentation** | Rotation, shift, zoom, flip (train only) | Reduces overfitting on small datasets |
| **Phase 1** | Frozen VGG16 base, train head at LR=1e-3 | Fast convergence, preserves ImageNet features |
| **Phase 2** | Unfreeze top 8 VGG layers at LR=1e-5 | Adapts high-level features to fingerprints |
| **Callbacks** | EarlyStopping + ReduceLR + Checkpoint | Prevents overfitting, auto-saves best weights |
| **Evaluation** | Accuracy + F1 + Confusion Matrix | Complete view of model strengths and errors |
| **Prediction fn** | Reusable function with confidence bars | Drop-in ready for Flask deployment |

### Tips for Higher Accuracy
- Collect more labelled fingerprint images (especially for minority classes like `AB-`, `O-`)
- Try **EfficientNetB0** or **MobileNetV2** as alternatives to VGG16 (often better accuracy with fewer parameters)
- Use **class_weight** in `model.fit()` if class imbalance is severe
- Add **OpenCV preprocessing** (histogram equalisation, Gabor filter) before feeding to the model